# 1.导入必要的深度学习Python包

In [1]:
# 检查当前安装的库版本
!pip show transformers accelerate peft

# 安装兼容的版本
!pip install transformers==4.33.0 accelerate==0.21.0 peft==0.4.0 datasets==2.14.0 --quiet

print("库版本已更新")

Name: transformers
Version: 4.53.3
Summary: State-of-the-art Machine Learning for JAX, PyTorch and TensorFlow
Home-page: https://github.com/huggingface/transformers
Author: The Hugging Face team (past and future) with the help of all our contributors (https://github.com/huggingface/transformers/graphs/contributors)
Author-email: transformers@huggingface.co
License: Apache 2.0 License
Location: /usr/local/lib/python3.11/dist-packages
Requires: filelock, huggingface-hub, numpy, packaging, pyyaml, regex, requests, safetensors, tokenizers, tqdm
Required-by: kaggle-environments, peft, sentence-transformers
---
Name: accelerate
Version: 1.9.0
Summary: Accelerate
Home-page: https://github.com/huggingface/accelerate
Author: The HuggingFace team
Author-email: zach.mueller@huggingface.co
License: Apache
Location: /usr/local/lib/python3.11/dist-packages
Requires: huggingface_hub, numpy, packaging, psutil, pyyaml, safetensors, torch
Required-by: peft
---
Name: peft
Version: 0.16.0
Summary: Paramet

In [2]:
import transformers
print(transformers.__version__)

4.33.0


In [3]:
# 基础库
import os
import json
import random

# 数据处理
import pandas as pd

# 深度学习框架
import torch
from torch.utils.data import Dataset, DataLoader

# 预训练模型
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer

# 参数高效微调
from peft import LoraConfig, get_peft_model, TaskType

# 进度显示
from tqdm.auto import tqdm

# 设置随机种子
random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

# 在导包部分添加
import accelerate
from accelerate import Accelerator

# 设置环境变量优化
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:128"

# 清空GPU缓存并设置优化标志
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.backends.cudnn.benchmark = True  # 加速卷积运算
    torch.backends.cuda.matmul.allow_tf32 = True  # 允许TF32
    torch.backends.cudnn.allow_tf32 = True  # 允许TF32

# 设置更安全的混合精度
torch.autocast("cuda", dtype=torch.float16).__enter__()

print("所有核心包导入完成！")

2025-12-01 12:37:31.024070: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764592651.180989      47 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764592651.230289      47 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

/usr/local/lib/python3.11/dist-packages/transformers/utils/generic.py:311: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_node(
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()`

所有核心包导入完成！


# 2.导入预训练模型与分词器

In [4]:
def load_pretrained_model_and_tokenizer_single_gpu(model_name="THUDM/chatglm2-6b"):
    """
    修复的加载预训练模型和分词器函数，确保在单个GPU上
    """
    try:
        print("开始加载分词器...")
        # 加载分词器
        tokenizer = AutoTokenizer.from_pretrained(
            model_name,
            trust_remote_code=True
        )
        
        # 检查并设置pad_token
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
            print(f"设置pad_token为eos_token: {tokenizer.eos_token}")
        
        print("✅ 分词器加载成功!")
        
        print("开始加载预训练模型...")
        
        # 检查可用的GPU数量
        if torch.cuda.is_available():
            num_gpus = torch.cuda.device_count()
            print(f"可用GPU数量: {num_gpus}")
            
            if num_gpus > 1:
                print("检测到多个GPU，将模型分配到单个GPU上...")
                # 修复：指定使用第一个GPU，避免自动分配到多个GPU
                model = AutoModelForCausalLM.from_pretrained(
                    model_name,
                    trust_remote_code=True,
                    torch_dtype=torch.float16,
                    device_map={"": 0}  # 强制使用第一个GPU
                )
            else:
                # 只有一个GPU，正常加载
                model = AutoModelForCausalLM.from_pretrained(
                    model_name,
                    trust_remote_code=True,
                    torch_dtype=torch.float16,
                    device_map="auto"
                )
        else:
            print("使用CPU模式")
            model = AutoModelForCausalLM.from_pretrained(
                model_name,
                trust_remote_code=True,
                torch_dtype=torch.float32,
                device_map=None
            )
        
        print("✅ 预训练模型加载成功!")
        
        # 检查模型基本信息
        print(f"模型名称: {model_name}")
        print(f"模型参数量: {sum(p.numel() for p in model.parameters()):,}")
        print(f"模型设备: {next(model.parameters()).device}")
        
        return tokenizer, model
        
    except Exception as e:
        print(f"❌ 加载失败: {str(e)}")
        import traceback
        traceback.print_exc()
        return None, None

# 重新执行修复的第二步
print("=" * 50)
print("第二步: 导入预训练模型与分词器（单GPU修复版）")
print("=" * 50)

tokenizer, model = load_pretrained_model_and_tokenizer_single_gpu()

if tokenizer is not None and model is not None:
    print("\n🎉 第二步完成: 预训练模型和分词器导入成功!")
    print(f"确认模型设备: {next(model.parameters()).device}")
else:
    print("\n💥 第二步失败: 请检查模型路径或网络连接")

第二步: 导入预训练模型与分词器（单GPU修复版）
开始加载分词器...


/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/244 [00:00<?, ?B/s]

tokenization_chatglm.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/THUDM/chatglm2-6b:
- tokenization_chatglm.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


...

pytorch_model-00005-of-00007.bin:   0%|          | 0.00/1.97G [00:00<?, ?B/s]

pytorch_model-00006-of-00007.bin:   0%|          | 0.00/1.93G [00:00<?, ?B/s]

pytorch_model-00007-of-00007.bin:   0%|          | 0.00/1.05G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/7 [00:00<?, ?it/s]

✅ 预训练模型加载成功!
模型名称: THUDM/chatglm2-6b
模型参数量: 6,243,584,000
模型设备: cuda:0

🎉 第二步完成: 预训练模型和分词器导入成功!
确认模型设备: cuda:0


# 3.数据导入与冻结权重，并配置参数高效微调方法（如LoRA）

In [5]:
def prepare_data_and_configure_lora_gpu(data_path='/kaggle/input/60000/Product_Review_Data.xlsx', 
                                       train_size=1000, test_size=200):
    """
    准备数据并配置LoRA参数高效微调，确保使用GPU
    """
    try:
        print("=" * 50)
        print("第三步: 数据导入与LoRA配置（GPU版本）")
        print("=" * 50)
        
        # 1. 数据导入与处理
        print("1. 导入和处理数据...")
        df = pd.read_excel(data_path)
        print(f"原始数据形状: {df.shape}")
        
        # 重命名列以符合我们的处理逻辑
        df = df.rename(columns={df.columns[0]: 'label', df.columns[1]: 'review'})
        
        # 数据清洗：移除空值和重复值
        df = df.dropna(subset=['review', 'label'])
        df = df.drop_duplicates(subset=['review'])
        print(f"清洗后数据形状: {df.shape}")
        
        # 2. 按照论文要求划分数据集
        print("2. 划分训练集和测试集...")
        # 随机选择1000条数据用于训练专家模型
        train_df = df.sample(n=train_size, random_state=42)
        
        # 剩余数据用于生成伪标签
        remaining_df = df.drop(train_df.index)
        
        # 从训练集中划分测试集（用于评估专家模型性能）
        test_df = train_df.sample(n=test_size, random_state=42)
        train_df_final = train_df.drop(test_df.index)
        
        print(f"训练集大小: {len(train_df_final)}")
        print(f"测试集大小: {len(test_df)}") 
        print(f"剩余数据大小(用于伪标签): {len(remaining_df)}")
        
        # 3. 配置LoRA参数高效微调
        print("3. 配置LoRA参数...")
        lora_config = LoraConfig(
            task_type=TaskType.CAUSAL_LM,
            inference_mode=False,
            r=8,                    # LoRA秩 - 论文指定
            lora_alpha=32,          # LoRA alpha - 论文指定  
            lora_dropout=0.1,       # LoRA dropout - 论文指定
            target_modules=["query_key_value"],  # ChatGLM2特定模块
            bias="none"
        )
        
        # 应用LoRA配置，自动冻结基础模型权重
        peft_model = get_peft_model(model, lora_config)
        
        # 修复：确保peft_model也在GPU上
        if torch.cuda.is_available():
            peft_model = peft_model.cuda()
        
        # 4. 检查配置结果
        print("4. 检查配置结果...")
        trainable_params = 0
        all_params = 0
        for _, param in peft_model.named_parameters():
            all_params += param.numel()
            if param.requires_grad:
                trainable_params += param.numel()
        
        print(f"可训练参数量: {trainable_params:,}")
        print(f"总参数量: {all_params:,}")
        print(f"可训练参数占比: {100 * trainable_params / all_params:.2f}%")
        print(f"PEFT模型设备: {next(peft_model.parameters()).device}")
        
        return train_df_final, test_df, remaining_df, peft_model, tokenizer
        
    except Exception as e:
        print(f"❌ 数据准备和LoRA配置失败: {str(e)}")
        import traceback
        traceback.print_exc()
        return None, None, None, None, None

# 重新执行第三步，确保使用GPU
train_data, test_data, remaining_data, peft_model, tokenizer = prepare_data_and_configure_lora_gpu()

if train_data is not None and peft_model is not None:
    print("\n🎉 第三步完成: 数据导入和LoRA配置成功!")
    print("✅ 数据集划分完成")
    print("✅ LoRA参数配置完成") 
    print("✅ 权重冻结完成")
    print(f"✅ PEFT模型设备确认: {next(peft_model.parameters()).device}")
else:
    print("\n💥 第三步失败: 请检查数据路径或配置参数")

第三步: 数据导入与LoRA配置（GPU版本）
1. 导入和处理数据...
原始数据形状: (62774, 2)
清洗后数据形状: (62724, 2)
2. 划分训练集和测试集...
训练集大小: 800
测试集大小: 200
剩余数据大小(用于伪标签): 61724
3. 配置LoRA参数...
4. 检查配置结果...
可训练参数量: 1,949,696
总参数量: 6,245,533,696
可训练参数占比: 0.03%
PEFT模型设备: cuda:0

🎉 第三步完成: 数据导入和LoRA配置成功!
✅ 数据集划分完成
✅ LoRA参数配置完成
✅ 权重冻结完成
✅ PEFT模型设备确认: cuda:0


# 4.使用指令微调数据对模型进行微调

# 4.1 构建指令数据集类

In [6]:
class InstructionDatasetSimple(Dataset):
    """
    简化但有效的实现，确保输入和标签维度一致
    """
    def __init__(self, dataframe, tokenizer, max_length=64):  # 减少长度
        self.data = dataframe
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.label_mapping = {1: "正面", 0: "负面"}
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        review = str(row['review']).strip()
        label = int(row['label'])
        
        # 简单处理：将标签作为额外的token添加到输入中
        instruction = f"请识别以下句子表达的情感：\n{review}\n情感："
        output_text = self.label_mapping[label]
        
        # 完整文本
        full_text = instruction + output_text
        
        # 编码
        encoding = self.tokenizer(
            full_text,
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
            return_tensors="pt"
        )
        
        # 对于因果语言模型，标签通常就是input_ids本身
        # 模型会自动处理shifted labels
        return {
            "input_ids": encoding["input_ids"].squeeze(),
            "attention_mask": encoding["attention_mask"].squeeze(),
            "labels": encoding["input_ids"].squeeze()  # 重要：设置为相同
        }

# 4.2 准备训练数据

In [7]:
def prepare_training_data_correct(train_data, test_data, tokenizer):
    """
    修复的数据准备函数
    """
    try:
        print("准备训练和测试数据集...")
        
        # 使用修复的数据集类
        train_dataset = InstructionDatasetSimple(train_data, tokenizer)
        test_dataset = InstructionDatasetSimple(test_data, tokenizer)
        
        print(f"训练集样本数: {len(train_dataset)}")
        print(f"测试集样本数: {len(test_dataset)}")
        
        # 检查一个样本的形状
        sample = train_dataset[0]
        print(f"输入形状: {sample['input_ids'].shape}")
        print(f"标签形状: {sample['labels'].shape}")
        
        # 确保形状相同
        if sample['input_ids'].shape != sample['labels'].shape:
            print(f"❌ 错误: 输入形状 {sample['input_ids'].shape} 与标签形状 {sample['labels'].shape} 不匹配")
            return None, None
        
        print("✅ 数据集创建成功，输入和标签维度匹配")
        
        return train_dataset, test_dataset
        
    except Exception as e:
        print(f"❌ 数据准备失败: {str(e)}")
        import traceback
        traceback.print_exc()
        return None, None

# 4.3 配置训练参数并执行微调

In [16]:
def fine_tune_expert_model_with_visible_loss(peft_model, train_dataset, test_dataset, tokenizer, output_dir="./expert_model_visible"):
    """
    可显示训练损失和验证损失的版本 - 融合多种衰减策略
    """
    try:
        peft_model.config.use_cache = False
        
        print("配置训练参数（融合多种学习率衰减策略）...")
        
        # 检查数据集
        if len(train_dataset) == 0:
            print("❌ 训练数据集为空")
            return None
        
        # 参数配置
        batch_size = 4
        gradient_accumulation = 1
        num_epochs = 1
        warmup_steps = 10
        
        total_samples = len(train_dataset)
        steps_per_epoch = max(1, total_samples // batch_size)
        total_steps = steps_per_epoch * num_epochs
        
        print(f"训练配置:")
        print(f"  - 总训练步数: ~{total_steps}")
        print(f"  - 初始学习率: 2e-5")
        print(f"  - 融合策略: 热身 → 多步长衰减 → 余弦退火")
        print(f"  - 评估频率: 每50步")
        print(f"  - 日志频率: 每10步")
        
        # 训练参数
        training_args = TrainingArguments(
            output_dir=output_dir,
            
            # 基础训练参数
            per_device_train_batch_size=batch_size,
            per_device_eval_batch_size=batch_size,
            gradient_accumulation_steps=gradient_accumulation,
            num_train_epochs=num_epochs,
            
            # 评估和日志设置
            evaluation_strategy="steps",
            eval_steps=50,
            save_strategy="steps",
            save_steps=100,
            
            # 详细的日志记录
            logging_strategy="steps",
            logging_steps=10,
            logging_dir="./logs",
            log_level="info",
            
            # 混合精度
            fp16=True,
            
            # 数据加载
            gradient_checkpointing=False,
            dataloader_num_workers=2,
            dataloader_pin_memory=False,
            remove_unused_columns=True,
            
            # 报告和优化
            report_to="none",
            
            # 其他
            seed=42,
            max_grad_norm=1.0,
            load_best_model_at_end=False,
            metric_for_best_model="loss",
            greater_is_better=False,
            
            # 进度条和显示设置
            disable_tqdm=False,
            prediction_loss_only=True,
        )
        
        print("创建Trainer...")
        
        from transformers import DataCollatorForLanguageModeling
        data_collator = DataCollatorForLanguageModeling(
            tokenizer=tokenizer,
            mlm=False,
        )
        
        # 自定义回调函数来监控训练和学习率
        from transformers import TrainerCallback
        
        class LossCallback(TrainerCallback):
            def on_log(self, args, state, control, logs=None, **kwargs):
                if logs:
                    if 'loss' in logs:
                        print(f"[Step {state.global_step}] 训练损失: {logs['loss']:.4f}")
                    if 'eval_loss' in logs:
                        print(f"[Step {state.global_step}] 验证损失: {logs['eval_loss']:.4f}")
                    if 'learning_rate' in logs:
                        print(f"[Step {state.global_step}] 学习率: {logs['learning_rate']:.6f}")
        
        # 创建自定义优化器和调度器（融合策略）
        from torch.optim.lr_scheduler import MultiStepLR, CosineAnnealingLR, SequentialLR, LinearLR
        from transformers import AdamW
        
        class CustomTrainer(Trainer):
            def create_optimizer_and_scheduler(self, num_training_steps: int):
                # 创建优化器
                self.optimizer = AdamW(
                    self.model.parameters(),
                    lr=2e-5,  # 初始学习率
                    weight_decay=0.01
                )
                
                # 定义各阶段的步数
                warmup_steps = 10
                multi_step_steps = min(200, num_training_steps // 2)
                
                # 第一阶段：线性预热
                warmup_scheduler = LinearLR(
                    self.optimizer,
                    start_factor=0.1,  # 从10%的学习率开始
                    end_factor=1.0,    # 线性增加到100%
                    total_iters=warmup_steps
                )
                
                # 第二阶段：多步长衰减
                multi_step_scheduler = MultiStepLR(
                    self.optimizer,
                    milestones=[warmup_steps + 50, warmup_steps + 150, warmup_steps + 300],
                    gamma=0.7,
                )
                
                # 第三阶段：余弦退火
                cosine_steps = max(100, num_training_steps - warmup_steps - multi_step_steps)
                cosine_scheduler = CosineAnnealingLR(
                    self.optimizer,
                    T_max=cosine_steps,
                    eta_min=1e-6,
                )
                
                # 组合调度器
                self.lr_scheduler = SequentialLR(
                    self.optimizer,
                    schedulers=[warmup_scheduler, multi_step_scheduler, cosine_scheduler],
                    milestones=[warmup_steps, warmup_steps + multi_step_steps]
                )
                
                print(f"学习率调度器配置:")
                print(f"  - 预热阶段: 0-{warmup_steps}步")
                print(f"  - 多步长衰减阶段: {warmup_steps}-{warmup_steps + multi_step_steps}步")
                print(f"  - 余弦退火阶段: {warmup_steps + multi_step_steps}-{num_training_steps}步")
        
        # 创建Trainer并添加回调
        trainer = CustomTrainer(
            model=peft_model,
            args=training_args,
            train_dataset=train_dataset,
            eval_dataset=test_dataset,
            tokenizer=tokenizer,
            data_collator=data_collator,
            callbacks=[LossCallback()],
        )
        
        print("开始训练（使用融合学习率衰减策略）...")
        train_result = trainer.train()
        
        # 保存模型
        trainer.save_model()
        tokenizer.save_pretrained(output_dir)
        print(f"✅ 模型已保存到: {output_dir}")
        
        print(f"训练完成，总训练步数: {train_result.global_step}")
        print(f"最终训练损失: {train_result.training_loss:.4f}")
        
        # 显示详细的训练历史和学习率变化
        print("\n训练历史和学习率变化:")
        history = trainer.state.log_history
        for i in range(0, len(history), max(1, len(history)//20)):  # 显示约20个点
            log = history[i]
            if 'loss' in log:
                step = log.get('step', 'N/A')
                loss = log.get('loss', 'N/A')
                lr = log.get('learning_rate', 'N/A')
                if isinstance(loss, (int, float)) and isinstance(lr, (int, float)):
                    print(f"  步骤 {step}: 损失={loss:.4f}, 学习率={lr:.6f}")
        
        return trainer
        
    except Exception as e:
        print(f"❌ 模型微调失败: {str(e)}")
        import traceback
        traceback.print_exc()
        return None

# 4.4 执行完整的第四步

In [17]:
def execute_step_four_fast_minimal():
    """
    快速版第四步执行 - 无参数版本，使用全局变量
    """
    try:
        print("=" * 60)
        print("第四步: 快速微调专家模型")
        print("=" * 60)
        
        # 检查必要组件（使用全局变量）
        required_vars = ['train_data', 'test_data', 'peft_model', 'tokenizer']
        missing_vars = [var for var in required_vars if var not in globals()]
        if missing_vars:
            print(f"❌ 缺少必要的变量: {missing_vars}")
            return False
        
        print(f"模型设备: {next(peft_model.parameters()).device}")
        
        # 1. 准备训练数据
        print("1. 准备训练数据...")
        train_dataset, test_dataset = prepare_training_data_correct(train_data, test_data, tokenizer)
        
        if train_dataset is None:
            return False
        
        # 2. 执行快速模型微调
        print("2. 执行快速模型微调...")
        trainer = fine_tune_expert_model_with_visible_loss(peft_model, train_dataset, test_dataset, tokenizer)
        
        if trainer is None:
            return False
        
        print("🎉 第四步完成: 专家模型快速微调成功!")
        return True
        
    except Exception as e:
        print(f"❌ 第四步执行失败: {str(e)}")
        import traceback
        traceback.print_exc()
        return False

# 执行快速训练（无参数调用）
print("开始执行快速训练...")
success = execute_step_four_fast_minimal()
if success:
    print("\n✅ 专家模型快速训练完成! 准备进入第五步生成伪标签!")
else:
    print("\n💥 快速训练失败，检查错误信息!")

Found safetensors installation, but --save_safetensors=False. Safetensors should be a preferred weights saving format due to security and performance reasons. If your model cannot be saved by safetensors please feel free to open an issue at https://github.com/huggingface/safetensors!
PyTorch: setting up devices


开始执行快速训练...
第四步: 快速微调专家模型
模型设备: cuda:0
1. 准备训练数据...
准备训练和测试数据集...
训练集样本数: 800
测试集样本数: 200
输入形状: torch.Size([64])
标签形状: torch.Size([64])
✅ 数据集创建成功，输入和标签维度匹配
2. 执行快速模型微调...
配置训练参数（融合多种学习率衰减策略）...
训练配置:
  - 总训练步数: ~200
  - 初始学习率: 2e-5
  - 融合策略: 热身 → 多步长衰减 → 余弦退火
  - 评估频率: 每50步
  - 日志频率: 每10步
创建Trainer...
开始训练（使用融合学习率衰减策略）...


/usr/local/lib/python3.11/dist-packages/accelerate/accelerator.py:427: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)
/usr/local/lib/python3.11/dist-packages/transformers/optimization.py:411: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/accelerate/accelerator.py:1301: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  new_forward = torch.cuda.amp.autocast(dtype=torch.float16)(model_forward_func)
***** Running training *****
  Num examples = 800
  Num Epochs = 1
  Instantaneous batch size per device = 4
  Total train batch size (w. parallel, distributed & acc

学习率调度器配置:
  - 预热阶段: 0-10步
  - 多步长衰减阶段: 10-110步
  - 余弦退火阶段: 110-200步


Step,Training Loss,Validation Loss
50,7.041400,7.058750
100,6.784800,7.058750
150,6.823000,7.058750
200,6.889100,7.058750


...

[Step 150] 训练损失: 6.8230
[Step 150] 学习率: 0.000014
[Step 150] 验证损失: 7.0588
[Step 160] 训练损失: 7.0902
[Step 160] 学习率: 0.000011
[Step 170] 训练损失: 6.8930
[Step 170] 学习率: 0.000008
[Step 180] 训练损失: 7.0867
[Step 180] 学习率: 0.000006
[Step 190] 训练损失: 6.8465
[Step 190] 学习率: 0.000003


***** Running Evaluation *****
  Num examples = 200
  Batch size = 4


[Step 200] 训练损失: 6.8891
[Step 200] 学习率: 0.000002
[Step 200] 验证损失: 7.0588


Saving model checkpoint to ./expert_model_visible/checkpoint-200
tokenizer config file saved in ./expert_model_visible/checkpoint-200/tokenizer_config.json
Special tokens file saved in ./expert_model_visible/checkpoint-200/special_tokens_map.json


Training completed. Do not forget to share your model on huggingface.co/models =)


Saving model checkpoint to ./expert_model_visible
tokenizer config file saved in ./expert_model_visible/tokenizer_config.json
Special tokens file saved in ./expert_model_visible/special_tokens_map.json
tokenizer config file saved in ./expert_model_visible/tokenizer_config.json
Special tokens file saved in ./expert_model_visible/special_tokens_map.json


✅ 模型已保存到: ./expert_model_visible
训练完成，总训练步数: 200
最终训练损失: 7.0129

训练历史和学习率变化:
  步骤 10: 损失=7.0289, 学习率=0.000015
  步骤 20: 损失=7.0039, 学习率=0.000020
  步骤 30: 损失=7.1719, 学习率=0.000020
  步骤 40: 损失=7.0992, 学习率=0.000020
  步骤 50: 损失=7.0414, 学习率=0.000020
  步骤 60: 损失=7.0570, 学习率=0.000020
  步骤 70: 损失=6.9453, 学习率=0.000020
  步骤 80: 损失=7.0824, 学习率=0.000014
  步骤 90: 损失=7.0980, 学习率=0.000014
  步骤 100: 损失=6.7848, 学习率=0.000014
  步骤 110: 损失=7.2496, 学习率=0.000014
  步骤 120: 损失=7.0062, 学习率=0.000020
  步骤 130: 损失=7.1328, 学习率=0.000019
  步骤 140: 损失=6.9270, 学习率=0.000017
  步骤 150: 损失=6.8230, 学习率=0.000014
  步骤 160: 损失=7.0902, 学习率=0.000011
  步骤 170: 损失=6.8930, 学习率=0.000008
  步骤 180: 损失=7.0867, 学习率=0.000006
  步骤 190: 损失=6.8465, 学习率=0.000003
  步骤 200: 损失=6.8891, 学习率=0.000002
🎉 第四步完成: 专家模型快速微调成功!

✅ 专家模型快速训练完成! 准备进入第五步生成伪标签!


# 清理内存部分

In [18]:
# GPU内存清理函数
import gc
import torch

def clear_gpu_memory():
    """
    清理GPU内存，释放所有未使用的缓存和变量
    """
    print("开始清理GPU内存...")
    
    # 清空PyTorch的CUDA缓存
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print("✅ 已清空PyTorch CUDA缓存")
    
    # 执行垃圾回收
    gc.collect()
    
    # 强制垃圾回收
    gc.collect()
    
    # 显示当前GPU内存使用情况
    if torch.cuda.is_available():
        gpu_memory = torch.cuda.memory_allocated() / 1024**3
        gpu_memory_reserved = torch.cuda.memory_reserved() / 1024**3
        print(f"GPU内存已分配: {gpu_memory:.2f} GB")
        print(f"GPU内存已保留: {gpu_memory_reserved:.2f} GB")
    
    print("✅ GPU内存清理完成")

def release_model_from_memory():
    """
    释放之前加载的模型占用的内存
    """
    global model, peft_model, trainer, expert_model
    
    print("释放模型占用的内存...")
    
    # 删除模型变量
    models_to_delete = ['model', 'peft_model', 'trainer', 'expert_model']
    
    for model_name in models_to_delete:
        if model_name in globals():
            try:
                # 将模型移到CPU
                model_obj = globals()[model_name]
                if hasattr(model_obj, 'cpu'):
                    model_obj.cpu()
                
                # 删除变量引用
                globals()[model_name] = None
                print(f"✅ 已释放 {model_name}")
            except Exception as e:
                print(f"⚠️ 释放 {model_name} 时出错: {str(e)}")
    
    # 强制垃圾回收
    gc.collect()
    torch.cuda.empty_cache()
    
    print("✅ 模型内存释放完成")

def set_cuda_memory_config():
    """
    设置CUDA内存配置以减少碎片
    """
    import os
    
    # 设置环境变量来减少内存碎片
    os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
    print("✅ 已设置 PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True")
    
    # 其他可能的优化设置
    os.environ['CUDA_LAUNCH_BLOCKING'] = '0'  # 禁用同步启动，提高并发
    print("✅ 已设置 CUDA_LAUNCH_BLOCKING=0")
    
    return True

def check_gpu_memory_status():
    """
    检查GPU内存状态
    """
    if not torch.cuda.is_available():
        print("❌ CUDA不可用，无法检查GPU内存")
        return
    
    # 获取GPU设备数量
    device_count = torch.cuda.device_count()
    print(f"可用GPU数量: {device_count}")
    
    for i in range(device_count):
        # 获取GPU名称
        gpu_name = torch.cuda.get_device_name(i)
        
        # 获取内存信息
        total_memory = torch.cuda.get_device_properties(i).total_memory / 1024**3
        allocated_memory = torch.cuda.memory_allocated(i) / 1024**3
        reserved_memory = torch.cuda.memory_reserved(i) / 1024**3
        free_memory = total_memory - allocated_memory
        
        print(f"\nGPU {i} ({gpu_name}):")
        print(f"  总内存: {total_memory:.2f} GB")
        print(f"  已分配: {allocated_memory:.2f} GB")
        print(f"  已保留: {reserved_memory:.2f} GB")
        print(f"  可用内存: {free_memory:.2f} GB")
        
        # 计算使用百分比
        used_percentage = (allocated_memory / total_memory) * 100
        print(f"  使用率: {used_percentage:.1f}%")
    
    return True

def optimize_memory_for_model_loading(model_size_gb=13):
    """
    为加载大型模型优化内存
    """
    if not torch.cuda.is_available():
        print("⚠️ CUDA不可用，无法优化GPU内存")
        return False
    
    print(f"为加载约{model_size_gb}GB的模型优化内存...")
    
    # 检查当前可用内存
    total_memory = torch.cuda.get_device_properties(0).total_memory / 1024**3
    allocated_memory = torch.cuda.memory_allocated(0) / 1024**3
    free_memory = total_memory - allocated_memory
    
    print(f"当前可用内存: {free_memory:.2f} GB")
    print(f"模型所需内存: ~{model_size_gb:.2f} GB")
    
    if free_memory < model_size_gb * 1.2:  # 需要额外20%的缓冲区
        print("⚠️ 可用内存不足，尝试释放更多内存...")
        
        # 执行完整的内存清理
        release_model_from_memory()
        clear_gpu_memory()
        
        # 再次检查内存
        allocated_memory = torch.cuda.memory_allocated(0) / 1024**3
        free_memory = total_memory - allocated_memory
        
        print(f"清理后可用内存: {free_memory:.2f} GB")
        
        if free_memory < model_size_gb:
            print(f"❌ 内存仍然不足！需要至少 {model_size_gb:.2f} GB，但只有 {free_memory:.2f} GB")
            print("建议：")
            print("1. 重启Kernel以完全释放内存")
            print("2. 使用更小的模型")
            print("3. 使用CPU加载模型")
            return False
    
    print("✅ 内存充足，可以加载模型")
    return True

# 代码解释：
# 1. clear_gpu_memory(): 清空PyTorch的CUDA缓存并执行垃圾回收
# 2. release_model_from_memory(): 释放之前加载的模型占用的内存
# 3. set_cuda_memory_config(): 设置环境变量来减少内存碎片
# 4. check_gpu_memory_status(): 检查GPU内存使用情况
# 5. optimize_memory_for_model_loading(): 为加载大型模型优化内存

# 执行注意事项：
# 1. 在执行5.1之前，先调用这些函数来释放内存
# 2. 如果内存仍然不足，可能需要重启Kernel
# 3. 可以按需调用不同的函数组合

# 使用示例：
def prepare_for_expert_model_loading():
    """
    准备加载专家模型的完整流程
    """
    print("=" * 60)
    print("准备加载专家模型 - 内存优化阶段")
    print("=" * 60)
    
    # 1. 检查GPU状态
    check_gpu_memory_status()
    
    # 2. 设置CUDA内存配置
    set_cuda_memory_config()
    
    # 3. 释放之前模型占用的内存
    release_model_from_memory()
    
    # 4. 清理GPU内存
    clear_gpu_memory()
    
    # 5. 检查是否满足加载条件（ChatGLM2-6B大约需要13GB）
    success = optimize_memory_for_model_loading(model_size_gb=13)
    
    if success:
        print("\n✅ 内存优化完成，可以开始加载专家模型")
        return True
    else:
        print("\n❌ 内存优化失败，无法加载专家模型")
        return False

# 执行内存优化准备
memory_ready = prepare_for_expert_model_loading()

准备加载专家模型 - 内存优化阶段
可用GPU数量: 1

GPU 0 (Tesla P100-PCIE-16GB):
  总内存: 15.89 GB
  已分配: 11.70 GB
  已保留: 13.24 GB
  可用内存: 4.19 GB
  使用率: 73.6%
✅ 已设置 PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True
✅ 已设置 CUDA_LAUNCH_BLOCKING=0
释放模型占用的内存...
✅ 已释放 model
✅ 已释放 peft_model
✅ 模型内存释放完成
开始清理GPU内存...
✅ 已清空PyTorch CUDA缓存
GPU内存已分配: 0.03 GB
GPU内存已保留: 0.08 GB
✅ GPU内存清理完成
为加载约13GB的模型优化内存...
当前可用内存: 15.85 GB
模型所需内存: ~13.00 GB
✅ 内存充足，可以加载模型

✅ 内存优化完成，可以开始加载专家模型


# 伪标签生成

## 5.1 加载微调后的专家模型

In [19]:
# 5.1 加载微调后的专家模型并配置生成参数
def load_finetuned_expert_model():
    """
    加载第四步微调完成的专家模型
    """
    try:
        print("=" * 60)
        print("第五步: 加载微调后的专家模型")
        print("=" * 60)
        
        # 检查模型是否已保存
        model_path = "/kaggle/working/expert_model_visible"
        if not os.path.exists(model_path):
            print(f"❌ 找不到微调模型路径: {model_path}")
            print("请确保第四步已成功完成并保存模型")
            return None, None
        
        print("加载微调后的专家模型...")
        
        # 加载基础模型
        from transformers import AutoModelForCausalLM, AutoTokenizer
        
        # 重新加载tokenizer
        tokenizer = AutoTokenizer.from_pretrained(
            "THUDM/chatglm2-6b",
            trust_remote_code=True
        )
        
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        
        # 加载基础模型
        base_model = AutoModelForCausalLM.from_pretrained(
            "THUDM/chatglm2-6b",
            trust_remote_code=True,
            torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
            device_map={"": 0} if torch.cuda.is_available() else None
        )
        
        # 加载PEFT适配器权重
        from peft import PeftModel
        peft_model = PeftModel.from_pretrained(base_model, model_path)
        
        # 合并权重（可选，如果需要推理速度更快）
        merged_model = peft_model.merge_and_unload()
        
        # 设置为评估模式
        merged_model.eval()
        
        if torch.cuda.is_available():
            merged_model = merged_model.cuda()
        
        print(f"✅ 专家模型加载成功!")
        print(f"模型设备: {next(merged_model.parameters()).device}")
        print(f"模型参数量: {sum(p.numel() for p in merged_model.parameters()):,}")
        
        return merged_model, tokenizer
        
    except Exception as e:
        print(f"❌ 加载专家模型失败: {str(e)}")
        import traceback
        traceback.print_exc()
        return None, None

# 代码解释：
# 1. 加载第四步微调保存的专家模型
# 2. 从原始模型开始，然后加载PEFT适配器权重
# 3. 合并权重以获得完整的推理模型
# 4. 设置为评估模式以提高推理效率

# 执行注意事项：
# 1. 确保第四步已成功执行并保存模型到"./expert_model_finetuned"目录
# 2. 如果GPU内存不足，可以考虑不合并权重，直接使用peft_model进行推理
# 3. 推理时使用model.eval()模式，避免梯度计算

# 执行加载专家模型
expert_model, tokenizer = load_finetuned_expert_model()

第五步: 加载微调后的专家模型
加载微调后的专家模型...


/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
loading file tokenizer.model from cache at /root/.cache/huggingface/hub/models--THUDM--chatglm2-6b/snapshots/d2e2d91789248536a747d9ce60642a336444186c/tokenizer.model
loading file added_tokens.json from cache at None
loading file special_tokens_map.json from cache at None
loading file tokenizer_config.json from cache at /root/.cache/huggingface/hub/models--THUDM--chatglm2-6b/snapshots/d2e2d91789248536a747d9ce60642a336444186c/tokenizer_config.json
loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--THUDM--chatglm2-6b/snapshots/d2e2d91789248536a747d9ce60642a336444186c/config.json
loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--THUDM

Loading checkpoint shards:   0%|          | 0/7 [00:00<?, ?it/s]

All model checkpoint weights were used when initializing ChatGLMForConditionalGeneration.

All the weights of ChatGLMForConditionalGeneration were initialized from the model checkpoint at THUDM/chatglm2-6b.
If your task is similar to the task the model of the checkpoint was trained on, you can already use ChatGLMForConditionalGeneration for predictions without further training.
Generation config file not found, using a generation config created from the model config.


✅ 专家模型加载成功!
模型设备: cuda:0
模型参数量: 6,243,584,000


## 5.2定义伪标签生成函数

In [20]:
# 5.2 定义伪标签生成函数
class PseudoLabelGenerator:
    """
    使用微调后的专家模型生成伪标签的类
    """
    def __init__(self, model, tokenizer, device="cuda" if torch.cuda.is_available() else "cpu"):
        self.model = model
        self.tokenizer = tokenizer
        self.device = device
        self.model.to(self.device)
        self.model.eval()  # 设置为评估模式
        
        # 指令模板（与训练时保持一致）
        self.instruction_template = "请识别以下句子表达的情感：\n{}\n情感："
        
        # 标签映射
        self.label_mapping = {"正面": 1, "负面": 0}
        self.reverse_mapping = {1: "正面", 0: "负面"}
    
    def generate_pseudo_label(self, text, max_length=50):
        """
        为单个文本生成伪标签
        """
        try:
            # 构建指令
            instruction = self.instruction_template.format(text)
            
            # 编码输入
            inputs = self.tokenizer(
                instruction,
                return_tensors="pt",
                truncation=True,
                max_length=128,
                padding=True
            )
            
            # 移动到设备
            inputs = {k: v.to(self.device) for k, v in inputs.items()}
            
            # 生成回答
            with torch.no_grad():  # 不计算梯度，提高推理速度
                outputs = self.model.generate(
                    **inputs,
                    max_new_tokens=10,  # 只需要生成标签，不需要太长
                    temperature=0.1,    # 低温度使输出更确定
                    do_sample=False,    # 不使用采样，使用贪心解码
                    pad_token_id=self.tokenizer.pad_token_id,
                    eos_token_id=self.tokenizer.eos_token_id
                )
            
            # 解码输出
            generated_text = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
            
            # 提取标签部分
            label_text = generated_text.replace(instruction, "").strip()
            
            # 将文本标签映射为数字
            if "正面" in label_text:
                pseudo_label = 1
            elif "负面" in label_text:
                pseudo_label = 0
            else:
                # 如果模型输出不符合预期，使用默认标签
                print(f"⚠️ 无法识别的输出: '{label_text}'，使用默认标签0")
                pseudo_label = 0
            
            return pseudo_label, label_text
            
        except Exception as e:
            print(f"生成伪标签时出错: {str(e)}")
            return 0, "错误"  # 出错时返回默认标签
    
    def batch_generate_pseudo_labels(self, texts, batch_size=16, progress_bar=True):
        """
        批量生成伪标签
        """
        results = []
        
        # 创建进度条
        if progress_bar:
            from tqdm.auto import tqdm
            pbar = tqdm(total=len(texts), desc="生成伪标签")
        
        # 分批处理
        for i in range(0, len(texts), batch_size):
            batch_texts = texts[i:i+batch_size]
            batch_results = []
            
            for text in batch_texts:
                pseudo_label, label_text = self.generate_pseudo_label(text)
                batch_results.append({
                    'text': text,
                    'pseudo_label': pseudo_label,
                    'label_text': label_text
                })
            
            results.extend(batch_results)
            
            if progress_bar:
                pbar.update(len(batch_texts))
            
            # 每处理100条数据打印一次进度
            if i > 0 and i % (batch_size * 10) == 0:
                print(f"已处理 {i}/{len(texts)} 条数据")
        
        if progress_bar:
            pbar.close()
        
        return results

# 代码解释：
# 1. 定义PseudoLabelGenerator类，封装伪标签生成逻辑
# 2. generate_pseudo_label方法为单个文本生成伪标签
# 3. batch_generate_pseudo_labels方法批量处理数据，提高效率
# 4. 使用与训练时相同的指令模板确保一致性
# 5. 添加错误处理机制，当模型输出不符合预期时使用默认标签

# 执行注意事项：
# 1. 批量大小batch_size应根据GPU内存调整，内存不足时减小batch_size
# 2. 生成过程中使用torch.no_grad()避免内存泄漏
# 3. 温度参数temperature设置为较低值（0.1）使输出更稳定

## 5.3 标注伪标签数据

In [21]:
# 5.3 准备未标注数据并生成伪标签
def prepare_unlabeled_data_and_generate_pseudo_labels():
    """
    准备未标注数据并使用专家模型生成伪标签
    """
    try:
        print("=" * 60)
        print("准备未标注数据并生成伪标签")
        print("=" * 60)
        
        # 检查未标注数据是否存在
        if 'remaining_data' not in globals():
            print("❌ 未找到remaining_data，请确保第三步已正确执行")
            return None
        
        if expert_model is None or tokenizer is None:
            print("❌ 专家模型或分词器未加载，请先执行5.1")
            return None
        
        print(f"未标注数据数量: {len(remaining_data)}")
        
        # 创建伪标签生成器
        generator = PseudoLabelGenerator(expert_model, tokenizer)
        
        # 提取文本数据
        unlabeled_texts = remaining_data['review'].astype(str).tolist()
        
        print("开始生成伪标签...")
        
        # 批量生成伪标签（根据内存调整batch_size）
        if torch.cuda.is_available():
            gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1024**3
            if gpu_memory >= 16:
                batch_size = 32
            elif gpu_memory >= 8:
                batch_size = 16
            else:
                batch_size = 8
            print(f"GPU内存: {gpu_memory:.1f}GB, 使用batch_size={batch_size}")
        else:
            batch_size = 4
            print("使用CPU，batch_size=4")
        
        # 生成伪标签
        pseudo_label_results = generator.batch_generate_pseudo_labels(
            unlabeled_texts, 
            batch_size=batch_size,
            progress_bar=True
        )
        
        print(f"✅ 伪标签生成完成! 共生成 {len(pseudo_label_results)} 个伪标签")
        
        # 统计标签分布
        label_counts = {}
        for result in pseudo_label_results:
            label = result['pseudo_label']
            label_counts[label] = label_counts.get(label, 0) + 1
        
        print("伪标签分布统计:")
        for label, count in label_counts.items():
            label_name = "乐观情绪" if label == 1 else "消极情绪"
            percentage = (count / len(pseudo_label_results)) * 100
            print(f"  {label_name}({label}): {count} 条 ({percentage:.2f}%)")
        
        return pseudo_label_results
        
    except Exception as e:
        print(f"❌ 生成伪标签失败: {str(e)}")
        import traceback
        traceback.print_exc()
        return None

# 代码解释：
# 1. 从remaining_data中提取未标注的文本数据
# 2. 根据GPU内存动态调整batch_size
# 3. 使用PseudoLabelGenerator批量生成伪标签
# 4. 统计生成的伪标签分布情况

# 执行注意事项：
# 1. 确保remaining_data变量已从第三步正确加载
# 2. 根据可用GPU内存调整batch_size，避免内存溢出
# 3. 生成过程中会显示进度条，可以监控进度

## 保存到excel当中

In [22]:
# 5.4 将伪标签数据保存为Excel文件
def save_pseudo_labels_to_excel(pseudo_label_results, output_path="./pseudo_labeled_data.xlsx"):
    """
    将生成的伪标签数据保存为Excel文件
    """
    try:
        print("=" * 60)
        print("保存伪标签数据到Excel")
        print("=" * 60)
        
        if pseudo_label_results is None or len(pseudo_label_results) == 0:
            print("❌ 没有伪标签数据可保存")
            return False
        
        # 转换为DataFrame
        df_pseudo_labels = pd.DataFrame(pseudo_label_results)
        
        # 重命名列，使其更清晰
        df_pseudo_labels = df_pseudo_labels.rename(columns={
            'text': 'review',
            'pseudo_label': 'label',
            'label_text': 'predicted_emotion'
        })
        
        # 添加数据来源标记
        df_pseudo_labels['data_source'] = 'pseudo_label'
        
        # 重新排列列顺序
        df_pseudo_labels = df_pseudo_labels[['review', 'label', 'predicted_emotion', 'data_source']]
        
        # 保存为Excel文件
        df_pseudo_labels.to_excel(output_path, index=False)
        
        print(f"✅ 伪标签数据已保存到: {output_path}")
        print(f"数据形状: {df_pseudo_labels.shape}")
        print("\n数据预览:")
        print(df_pseudo_labels.head())
        
        # 保存统计信息
        stats_path = "./pseudo_label_statistics.txt"
        with open(stats_path, 'w', encoding='utf-8') as f:
            f.write("伪标签数据统计信息\n")
            f.write("=" * 50 + "\n")
            f.write(f"总数据量: {len(df_pseudo_labels)}\n")
            f.write("\n标签分布:\n")
            
            label_counts = df_pseudo_labels['label'].value_counts()
            for label, count in label_counts.items():
                label_name = "乐观情绪" if label == 1 else "消极情绪"
                percentage = (count / len(df_pseudo_labels)) * 100
                f.write(f"  {label_name}({label}): {count} 条 ({percentage:.2f}%)\n")
        
        print(f"✅ 统计信息已保存到: {stats_path}")
        
        return True
        
    except Exception as e:
        print(f"❌ 保存伪标签数据失败: {str(e)}")
        import traceback
        traceback.print_exc()
        return False

# 代码解释：
# 1. 将伪标签结果转换为pandas DataFrame
# 2. 重命名列使其更清晰易懂
# 3. 添加data_source列标记数据来源
# 4. 保存为Excel文件，方便后续使用
# 5. 同时保存统计信息文件，记录标签分布

# 执行注意事项：
# 1. 确保有写入权限到输出目录
# 2. Excel文件将包含原始文本、伪标签、预测的情感文本和数据来源
# 3. 保存的Excel文件可以直接用于后续的多模态模型训练

## 主程序

In [ ]:
# 5.5 主执行函数 - 执行完整的第五步
def execute_step_five():
    """
    执行完整的第五步：使用微调后的专家模型为未标注数据生成伪标签
    """
    try:
        print("=" * 80)
        print("开始执行第五步：生成伪标签")
        print("=" * 80)
        
        # 1. 加载专家模型
        print("\n1. 加载专家模型...")
        global expert_model, tokenizer
        if 'expert_model' not in globals() or expert_model is None:
            expert_model, tokenizer = load_finetuned_expert_model()
            if expert_model is None:
                return False
        
        # 2. 生成伪标签
        print("\n2. 生成伪标签...")
        pseudo_label_results = prepare_unlabeled_data_and_generate_pseudo_labels()
        if pseudo_label_results is None:
            return False
        
        # 3. 保存伪标签数据
        print("\n3. 保存伪标签数据...")
        success = save_pseudo_labels_to_excel(pseudo_label_results)
        if not success:
            return False
        
        print("\n" + "=" * 80)
        print("✅ 第五步完成：伪标签生成和保存成功!")
        print("=" * 80)
        
        # 显示下一步建议
        print("\n📋 下一步建议:")
        print("1. 伪标签数据已保存为 './pseudo_labeled_data.xlsx'")
        print("2. 可以使用这些伪标签训练多模态情感识别模型")
        print("3. 建议对伪标签数据进行质量检查，确保标签准确性")
        
        return True
        
    except Exception as e:
        print(f"❌ 第五步执行失败: {str(e)}")
        import traceback
        traceback.print_exc()
        return False

# 代码解释：
# 1. 整合所有第五步的功能，按顺序执行
# 2. 提供完整的错误处理和进度报告
# 3. 执行成功后给出下一步建议

# 执行注意事项：
# 1. 这是一个完整的执行流程，会依次执行加载模型、生成伪标签、保存数据
# 2. 如果中途失败，会显示详细的错误信息
# 3. 执行前确保前四步已成功完成

# 执行第五步
if __name__ == "__main__":
    success = execute_step_five()
    if success:
        print("\n🎉 恭喜！伪标签生成流程全部完成！")
    else:
        print("\n💥 伪标签生成流程失败，请检查错误信息")

开始执行第五步：生成伪标签

1. 加载专家模型...

2. 生成伪标签...
准备未标注数据并生成伪标签
未标注数据数量: 61724
开始生成伪标签...
GPU内存: 15.9GB, 使用batch_size=16


生成伪标签:   0%|          | 0/61724 [00:00<?, ?it/s]

Generate config GenerationConfig {
  "_from_model_config": true,
  "eos_token_id": 2,
  "pad_token_id": 0,
  "transformers_version": "4.33.0"
}

/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:362: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.1` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
Generate config GenerationConfig {
  "_from_model_config": true,
  "eos_token_id": 2,
  "pad_token_id": 0,
  "transformers_version": "4.33.0"
}



⚠️ 无法识别的输出: '请识别以下句子表达的情感：
﻿做父母一定要有刘墉这样的心态，不断地学习，不断地进步，不断地给自己补充新鲜血液，让自己保持一颗年轻的心。我想，这是他能很好的和孩子沟通的一个重要因素。读刘墉的文章，总能让我看到一个快乐的平易近人的父亲，他始终站在和孩子同样的高度，给孩子创造着一个充满爱和自由的生活环境。很喜欢刘墉在字里行间流露出的做父母的那种小狡黠，让人总是忍俊不禁，父母和子女之间有时候也是一种战斗，武力争斗过于低级了，智力较量才更有趣味。所以，做父母一定要有刘墉这样的心态，不断地'，使用默认标签0


Generate config GenerationConfig {
  "_from_model_config": true,
  "eos_token_id": 2,
  "pad_token_id": 0,
  "transformers_version": "4.33.0"
}



...

⚠️ 无法识别的输出: '作者力从马克思注意经济学角度来剖析当代'，使用默认标签0


Generate config GenerationConfig {
  "_from_model_config": true,
  "eos_token_id": 2,
  "pad_token_id": 0,
  "transformers_version": "4.33.0"
}

Generate config GenerationConfig {
  "_from_model_config": true,
  "eos_token_id": 2,
  "pad_token_id": 0,
  "transformers_version": "4.33.0"
}



⚠️ 无法识别的输出: '作者更多的是从圆圆母亲的角度来写这个文章'，使用默认标签0


Generate config GenerationConfig {
  "_from_model_config": true,
  "eos_token_id": 2,
  "pad_token_id": 0,
  "transformers_version": "4.33.0"
}



⚠️ 无法识别的输出: '作者对于某些电影的分析入木三分，为'，使用默认标签0
